# 03 · Feature engineering (Phase 4)

One row per 100 m grid cell with every PRD §9.3 feature (contract C4), for each feature year:

1. build the features (raster C2 + vector C3 → C4) and time the build;
2. summary table: rows, dropped cells, ranges, NaN share;
3. a map of each feature;
4. histograms;
5. correlation matrix of the model inputs, with |r| > 0.9 pairs noted and a decision on dropping.

Correctness is checked independently in `scripts/verify_features.py` (C2) and `scripts/verify_vector_features.py` (C3).

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import load_config
from src.features import schema
from src.features.build import write_feature_tables
from src.features.distance import write_vector_features
from src.features.grid import lattice
from src.features.raster_features import features_years, write_raster_features
from src.io_utils import setup_logging

cfg = load_config()
setup_logging(cfg, "features")
years = list(features_years(cfg))
print(cfg["aoi"]["name"], years)

## 1. Build (timed from the processed layers)

Rebuilds C2, C3 and C4 from scratch so the time is the full feature build (gate: < 10 min).

In [ ]:
t0 = time.perf_counter()
write_raster_features(cfg)
t1 = time.perf_counter()
write_vector_features(cfg)
t2 = time.perf_counter()
tables = write_feature_tables(cfg)
t3 = time.perf_counter()
print(f"raster C2 {t1 - t0:.0f} s, vector C3 {t2 - t1:.0f} s, merge C4 {t3 - t2:.0f} s")
print(f"total {(t3 - t0) / 60:.1f} min")

## 2. Summary

Which OSM snapshot each year uses (time-travel rule: 2018 features use the 2018 snapshot), cells kept and dropped (> 50 % nodata).

In [ ]:
grid = schema.read_contract(schema.contract_path(cfg, "C1"), "C1")
for y, df in tables.items():
    print(f"{y}: OSM snapshot {schema.vector_snapshot_for(cfg, y)}, {len(df):,} cells "
          f"({len(grid) - len(df)} dropped), cell_id unique: {df['cell_id'].is_unique}")
FEATS = [c for c in tables[years[0]].columns if c not in ("cell_id", "nodata_frac")]
summ = tables[years[0]][FEATS].describe(percentiles=[0.5]).T
summ["nan %"] = 100 * tables[years[0]][FEATS].isna().mean()
summ["model input"] = [c in schema.MODEL_INPUTS for c in FEATS]
summ["leaky"] = [c in schema.LEAKY_FEATURES for c in FEATS]
summ.round(3)

Own-cell LULC fractions must sum to 1:

In [ ]:
for y, df in tables.items():
    s = df[list(schema.FEATURE_GROUPS["lulc_own"])].sum(axis=1)
    print(y, f"fraction sums {s.min():.4f} .. {s.max():.4f}")

## 3. Feature maps (baseline year)

Distances are shown in metres (`expm1` of the stored `log1p`).

In [ ]:
lat = lattice(cfg)
base = tables[years[0]].merge(grid[["cell_id", "row", "col"]], on="cell_id")


def to_img(df, col):
    img = np.full(lat.shape, np.nan)
    v = df[col].to_numpy()
    img[df["row"], df["col"]] = np.expm1(v) if col.startswith("log_dist") else v
    return img


n = len(FEATS)
ncol = 5
fig, axes = plt.subplots(int(np.ceil(n / ncol)), ncol, figsize=(4 * ncol, 3.8 * np.ceil(n / ncol)))
for ax, col in zip(axes.flat, FEATS, strict=False):
    img = to_img(base, col)
    is_dist = col.startswith("log_dist")
    cmap = "magma_r" if is_dist else "viridis"
    im = ax.imshow(img, cmap=cmap, vmax=np.nanpercentile(img, 99.5))
    ax.set_title(col.replace("log_dist", "dist") + (" (m)" if is_dist else ""), fontsize=10)
    ax.set_axis_off()
    fig.colorbar(im, ax=ax, shrink=0.7)
for ax in axes.flat[n:]:
    ax.set_axis_off()
fig.suptitle(f"C4 features, {years[0]}")
plt.tight_layout()

Change between the feature years for the main context features (latest − baseline):

In [ ]:
last = tables[years[-1]].merge(grid[["cell_id", "row", "col"]], on="cell_id")
cols = ["ring500_built", "log_dist_built", "road_density", "log_dist_any"]
fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), 4.5))
for ax, col in zip(axes, cols, strict=True):
    d = to_img(last, col) - to_img(base, col)
    v = np.nanpercentile(np.abs(d), 99)
    im = ax.imshow(d, cmap="RdBu_r", vmin=-v, vmax=v)
    name = col.replace("log_dist", "dist") + (" (m)" if col.startswith("log_dist") else "")
    ax.set_title(f"{name}: {years[-1]} − {years[0]}", fontsize=10)
    ax.set_axis_off()
    fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()

**Reading the change maps.** Built-up context grows mainly in the south-east and south (the growth the validation run has to predict). Road density rises almost everywhere, including the long-built urban core in the north, by up to ~20 km/km²: most of that is OSM mapping completeness (the 2018 snapshot has far fewer residential streets mapped), not new roads. That's why the validation run pairs 2018 LULC with the 2018 OSM snapshot (time-travel rule), and why road features shouldn't be compared across snapshots as if they measured construction. A few cells lose built-up share (blue): ESRI class noise between years.

## 4. Histograms

In [ ]:
fig, axes = plt.subplots(int(np.ceil(n / ncol)), ncol, figsize=(4 * ncol, 2.8 * np.ceil(n / ncol)))
for ax, col in zip(axes.flat, FEATS, strict=False):
    for y in years:
        ax.hist(tables[y][col], bins=50, histtype="step", label=str(y), density=True)
    ax.set_title(col, fontsize=10)
    ax.set_yticks([])
for ax in axes.flat[n:]:
    ax.set_axis_off()
axes.flat[0].legend()
plt.tight_layout()

## 5. Correlation of the model inputs

Pearson r over all cells, baseline year (`schema.MODEL_INPUTS`: no own-cell or leaky features).

In [ ]:
X = tables[years[0]][list(schema.MODEL_INPUTS)]
r = X.corr()
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(r, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(r)), r.columns, rotation=90)
ax.set_yticks(range(len(r)), r.columns)
for i in range(len(r)):
    for j in range(len(r)):
        ax.text(j, i, f"{r.iloc[i, j]:.2f}", ha="center", va="center", fontsize=6)
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()

In [ ]:
iu = np.triu_indices(len(r), 1)
pairs = pd.DataFrame({"a": r.columns[iu[0]], "b": r.columns[iu[1]], "r": r.to_numpy()[iu]})
pairs.reindex(pairs["r"].abs().sort_values(ascending=False).index).head(12).round(3)

**Highly correlated features (|r| > 0.9) and decision**

| pair | r | decision |
|---|---|---|
| ring250_crop / ring500_crop | 0.95 | keep both in C4 |
| slope_mean / slope_max | 0.95 | keep both in C4; clustering (P5.1) uses `slope_mean` only |
| ring250_built / ring500_built | 0.95 | keep both in C4 |
| ring250_range / ring500_range | 0.92 | keep both in C4 |
| ring250_tree / ring500_tree | 0.91 | keep both in C4 |

- Nothing is dropped from the feature table: it's a contract (C4) shared by every model, and the RF is insensitive to correlated inputs.
- The 250 m / 500 m pairs are the same class at two neighbourhood scales; r ≤ 0.95 means the scales still differ in places (edges of built-up areas, small patches). In the Euclidean similarity (P6) the pairs give each class's context about twice the weight of a single feature. P6.6 (held-out buildings) will compare all inputs vs 500 m rings only and keep the better one.
- `frac_snow` is constant 0 and `frac_flooded`, `frac_bare` are almost always 0: drop them from the clustering inputs (P5.1). They're own-cell fractions, so models never see them anyway.